# Planetary Analogs — בדיקת דאטה לפני מידול
הריצו את כל התאים לפי הסדר. ברירת המחדל מורידה את המאגר המורחב מ־GitHub. אפשר להעלות ל־Colab את העותק המקומי דרך חלונית Files ולהגדיר את הנתיב שלו ב־`LOCAL_DB`. [מקורות ומילון שדות](https://github.com/romimana/Planetary-analogs/blob/main/ML.md).

In [ ]:
import hashlib
import json
import sqlite3
from contextlib import closing
from pathlib import Path
from urllib.request import urlopen

import pandas as pd
from IPython.display import display

DATA_REF = "main"  # Set a commit SHA to reproduce a fixed release.
LOCAL_DB = None  # Example: "/content/ml-enriched-local.sqlite"

if LOCAL_DB:
    db_path = Path(LOCAL_DB).resolve()
else:
    base = f"https://raw.githubusercontent.com/romimana/Planetary-analogs/{DATA_REF}/data"
    with urlopen(f"{base}/ml-enriched.json", timeout=60) as response:
        expected_hash = json.load(response)["sha256"]
    with urlopen(f"{base}/ml-enriched.sqlite", timeout=120) as response:
        content = response.read()
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise ValueError("Checksum mismatch. Re-run this cell or use a pinned DATA_REF.")
    db_path = Path("ml-enriched.sqlite").resolve()
    db_path.write_bytes(content)

with closing(sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)) as db:
    objects = db.execute(
        "SELECT name, type FROM sqlite_schema WHERE type IN ('table', 'view') "
        "AND name NOT LIKE 'sqlite_%' ORDER BY type, name"
    ).fetchall()
    tables = {
        name: pd.read_sql_query('SELECT * FROM "' + name.replace('"', '""') + '"', db)
        for name, _ in objects
    }

inventory = pd.DataFrame([
    {"name": name, "type": kind, "rows": len(tables[name]), "columns": tables[name].shape[1]}
    for name, kind in objects
])
display(inventory)

## תצוגה וחוסרים
שנו את `TABLE` ורוצו שוב. שמות הטבלאות מופיעים בתא הקודם. אפשר לגשת לכל טבלה גם דרך `tables["name"]`.

In [ ]:
TABLE = "earth_model_profiles"
df = tables[TABLE]
display(df.head(8))
missing = df.isna().sum()
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": missing,
    "missing_pct": (missing / max(len(df), 1) * 100).round(1),
}).sort_values("missing", ascending=False))

## הכנת פיצ׳רים לדירוג
ברירת המחדל משתמשת בארבעה רכיבי כימיה ובטופוגרפיה. השדות האופציונליים נשלטים בתא הבא. X_earth_ready ו־X_targets_ready הן המטריצות המסודרות לדירוג. קודי מסלע נשארים בטבלאות הפרופיל לסינון ולעיון.

In [ ]:
import numpy as np

BODY = "mars"  # "mars" or "moon"
INCLUDE_SILICA = False
INCLUDE_MANGANESE = False
REQUIRE_MAPPED_GEOLOGY = False
# Leave both None for static matching. To include the Sun, pick one listed UTC per body.
EARTH_SOLAR_UTC = None
TARGET_SOLAR_UTC = None

CHEMISTRY = {
    "tio2_wt_pct": "Titanium dioxide",
    "mgo_wt_pct": "Magnesium oxide",
    "cao_wt_pct": "Calcium oxide",
    "k2o_wt_pct": "Potassium oxide",
    "sio2_wt_pct": "Silica",
    "mno_wt_pct": "Manganese oxide",
}
TERRAIN = [
    "slope_median_deg", "slope_p90_deg", "slope_over_10_deg_fraction",
    "relief_p90_p10_m", "local_roughness_rms_m", "largest_gentle_patch_fraction",
]
# Edit this list to use a specific subset of terrain measures.
terrain_features = TERRAIN.copy()
exclude = ({"sio2_wt_pct"} if not INCLUDE_SILICA else set()) | (
    {"mno_wt_pct"} if not INCLUDE_MANGANESE else set()
)
chem_features = [name for name in CHEMISTRY if name not in exclude]
features = [*chem_features, *terrain_features]
if BODY not in {"mars", "moon"}:
    raise ValueError("BODY must be 'mars' or 'moon'")
earth = tables["earth_model_profiles"].copy()
targets = tables[f"{BODY}_model_profiles"].copy()
solar = tables["solar_scenarios"]
display(solar.loc[solar["body"].isin(["earth", BODY]), ["body", "utc"]].drop_duplicates().reset_index(drop=True))

def join_solar(profile, body, key, utc):
    scenario = solar.loc[(solar["body"] == body) & (solar["utc"] == utc),
                         ["location_id", "horizontal_projection"]]
    if scenario.empty:
        raise ValueError(f"No published {body} solar scenario for {utc}")
    return profile.merge(scenario, left_on=key, right_on="location_id", how="left", validate="many_to_one")

if EARTH_SOLAR_UTC is not None or TARGET_SOLAR_UTC is not None:
    if EARTH_SOLAR_UTC is None or TARGET_SOLAR_UTC is None:
        raise ValueError("Select a solar scenario for both bodies")
    earth = join_solar(earth, "earth", "site_id", EARTH_SOLAR_UTC)
    targets = join_solar(targets, BODY, "id", TARGET_SOLAR_UTC)
    features.append("horizontal_projection")

required = features + (["geology_unit"] if REQUIRE_MAPPED_GEOLOGY else [])
earth_model = earth.dropna(subset=required).set_index("id", verify_integrity=True)
target_model = targets.dropna(subset=required).set_index("id", verify_integrity=True)
X_earth = earth_model[features].copy()
X_targets = target_model[features].copy()
if X_earth.empty or X_targets.empty:
    raise ValueError("The selected fields and filters leave no eligible candidates or targets")
for x in (X_earth, X_targets):
    if not np.isfinite(x.to_numpy(dtype=float)).all() or x[chem_features].le(0).any().any():
        raise ValueError("Invalid value in selected features")

def centered_log_ratios(frame):
    transformed = frame.copy()
    logs = np.log(frame[chem_features])
    transformed[chem_features] = logs.sub(logs.mean(axis=1), axis=0)
    return transformed.rename(columns={key: f"clr_{key}" for key in chem_features})

earth_vectors = centered_log_ratios(X_earth)
target_vectors = centered_log_ratios(X_targets)
center = earth_vectors.median()
scale = earth_vectors.quantile(.75) - earth_vectors.quantile(.25)
# Keep zero-IQR features on their native transformed units so weights remain explicit.
scale = scale.mask(scale <= 1e-12, 1.0)
X_earth_ready = earth_vectors.sub(center).div(scale)
X_targets_ready = target_vectors.sub(center).div(scale)
feature_scaling = pd.DataFrame({"earth_median": center, "divisor": scale})
display(earth_model.groupby("source_id").size().rename("eligible_samples"))
display(feature_scaling)
display(X_earth_ready.head())
display(X_targets_ready.head())

## תרחישים סביבתיים
טבלאות `complete_earth`, `complete_mars`, `complete_moon` הן נקודת הפתיחה כשצריך תרחיש מלא. הטמפרטורות של MODIS, REMS ו־Diviner תלויות בשיטת המדידה. עיינו בשם השדה ובתיעוד לפני השוואה. טבלאות המקור משמרות מדידות אוויר ופני שטח עם חותמות זמן נפרדות. שמרו תרחישים מאותו אתר יחד בזמן הערכה.

In [ ]:
earth_scenarios = tables["complete_earth"]
target_scenarios = tables[f"complete_{BODY}"]
display(earth_scenarios[["site_id", "date", "pass", "air_temperature_k",
                         "temperature_k", "pressure_pa", "weather_distance_m"]].head())
temperature = "ground_brightness_k" if BODY == "mars" else "lunar_surface_bolometric_k"
display(target_scenarios[["id", "window_label", "local_solar_hour", "distance_m", temperature]].head())

## כיסוי כל השכבות לפי דגימה
`point_profiles["earth"]`, `point_profiles["mars"]`, `point_profiles["moon"]` מציגות את כל הדגימות הזמינות למידול. עמודות שמסתיימות ב־`_n` סופרות תצפיות. הערכים עצמם נמצאים ב־`earth_weather`, ב־`tables["earth_modis"]` וב־`tables["planetary_profiles"]`. השארנו את סדרות הזמן נפרדות כדי למנוע צירוף בין שעות שונות.
זווית השמש נשמרת ב־`solar_scenarios` לכל שלושת העולמות. חברו לפי `body` ולפי `location_id` (מזהה אתר בכדור הארץ, מזהה דגימה במאדים ובירח). תרחישי השמש מוגדרים מראש ומתוארכים. [מילון השדות והמקורות](https://github.com/romimana/Planetary-analogs/blob/main/DATA_CODEBOOK.md)

In [ ]:
earth_weather = tables["earth_environment"]
solar = tables["solar_scenarios"]

# Keep full scenarios separate. These per-sample profiles show availability without multiplying rows.
point_profiles = {
    "earth": tables["earth_model_profiles"].copy(),
    "mars": tables["mars_model_profiles"].copy(),
    "moon": tables["moon_model_profiles"].copy(),
}

def add_counts(profile, key, observations, observation_key, fields):
    grouped = observations.groupby(observation_key)
    for field in fields:
        profile[field + "_n"] = profile[key].map(grouped[field].count()).fillna(0).astype(int)

earth = point_profiles["earth"]
add_counts(earth, "site_id", earth_weather, "site_id",
           ["air_temperature_k", "pressure_pa", "wind_speed_10m_m_s", "soil_water_0_to_7cm_m3_m3"])
add_counts(earth, "site_id", tables["earth_modis"], "site_id", ["temperature_k"])
earth.rename(columns={"temperature_k_n": "surface_temperature_n"}, inplace=True)
add_counts(earth, "site_id", tables["earth_radar"], "site_id", ["median_dn"])

for body in ("mars", "moon"):
    station = tables["planetary_profiles"].loc[
        lambda df: (df["body"] == body) & (df["comparison_status"] == "rover_or_station")
    ]
    surface = "ground_brightness_k" if body == "mars" else "lunar_surface_bolometric_k"
    fields = [surface, "local_solar_hour"]
    if body == "mars":
        fields += ["air_temperature_k", "pressure_pa"]
    add_counts(point_profiles[body], "id", station, "id", fields)
    point_profiles[body].rename(columns={surface + "_n": "surface_temperature_n"}, inplace=True)

for body, profile in point_profiles.items():
    key = "site_id" if body == "earth" else "id"
    add_counts(profile, key, solar[solar["body"] == body], "location_id", ["solar_zenith_deg"])

add_counts(earth, "id", tables["earth_grain_notes"], "sample_id", ["source_grain_description_ja"])
add_counts(point_profiles["mars"], "id", tables["mars_texture_context"], "sample_id", ["texture_context"])
exact_grain = tables["moon_grain_bins"].query("join_level == 'same_aliquot'")
add_counts(point_profiles["moon"], "id", exact_grain, "sample_id", ["weight_pct"])
mars_wind = tables["mars_environment"].query("comparison_status == 'rover_or_station'")
add_counts(point_profiles["mars"], "id", mars_wind, "id", ["wind_speed_m_s"])

audit = pd.DataFrame({
    body: pd.Series({
        column: int((frame[column] == 0).sum() if column.endswith("_n") else frame[column].isna().sum())
        for column in frame
    }, dtype="Int64")
    for body, frame in point_profiles.items()
})
display(audit.loc[[*CHEMISTRY, *TERRAIN, "geology_unit", "surface_temperature_n",
                   "air_temperature_k_n", "pressure_pa_n", "solar_zenith_deg_n",
                   "local_solar_hour_n", "median_dn_n",
                   "wind_speed_10m_m_s_n", "soil_water_0_to_7cm_m3_m3_n", "wind_speed_m_s_n",
                   "source_grain_description_ja_n", "texture_context_n", "weight_pct_n"]])
display(point_profiles[BODY].head())
display(earth_weather.head())